# FM study · re-tests RT1 and RT2 (post hoc)

**This notebook cannot run while it pins commit `000…0`.** Section 13 of `protocols.md` is a draft rule.
After Lam approves it, the rule and this code are pushed and an agent rebuilds the notebook with both
commits; Step 2a then checks them before anything is downloaded.

One session on an **A100 or L4**, about 2.5 h (an estimate from the Phase-1 timings, not measured): setup up
to 1 h; RT2 about 70 min (Inception-v3 needs about 34 min for all 5,927 images, and there are two sets); RT1
about 20 min. It reads no label and computes no score. Scoring runs once, on the laptop.

Fill in **RUN_BY**. Add `HF_TOKEN` to Colab Secrets and grant this notebook access (RETFound is gated), keep
the `octfsl` shortcut in My Drive, then **Runtime → Run all**.

| Step | What it does | What proves it worked |
|---|---|---|
| 0 | Checks the token, the RETFound page and Drive | The `RETEST PREFLIGHT` line |
| 1–2 | Checks the GPU, clones the pinned code, builds the locked Python environment | Commit matches; `pip check` passes |
| 2a | Rules before numbers: the pinned commit is on `origin` and holds the approved rule | `rule ok` |
| 3–6 | Unpacks the images, restores both locks, downloads RETFound, Inception-v3, OCTID and OCTDL | Every SHA256 and MD5 matches |
| 6c | Repeats the checks in one line | The `RETEST CHECKS` line |
| 7–8 | Verifies earlier saves; decodes every image with PIL and saves it as PNG (RT2 inputs) | `render` saved |
| 9 | RT2 first (I-pil, then the control I-ref), then RT1 (R-ref, the upstream check, three read-outs) | Each worker saved before the next starts |
| 10 | Saves the logs and prints the stamp | The `RETEST STAMP` line |

Keep `DROP_LEVEL = 0`. With `DROP_LEVEL = 1`, RT2 leaves out OCTID and OCTDL (about 10 min less) and its
nine-way table is not scored. Choose before the first run: a second run does not add them.

Every completed worker is archived with a SHA256 receipt as soon as it exits. Run a failed step once more
with the same settings. If it fails again, run **Save logs and stamp** and report the failure. The
destination is `octfsl/fm_v1/final/final-20261113T170804Z/retest/`. Send the **RETEST STAMP** line, any
retries and their reasons, and confirmation of the Drive files.

In [ ]:
# Fill in RUN_BY before Run all
RUN_BY = ""  # your name
ATTEMPT = "final-20261113T170804Z"  # the locked attempt; do not change
LOCK_SHA256 = "239838279b77bb37211398a016c1c766273a35f5a06649b31f5cd97f562c001c"
EXTERNAL_SHA256 = "3414889b53ee10c526c4d3dfb28b7477676f5bf449c2729008902f93447f45d0"
# Used by the reused external-download cell only; copied verbatim from the locked notebook's first cell.
EXTERNAL_DECISION = ('Lam, 2026-11-11: "Approve" (download OCTID and OCTDL; cohorts fixed by amendment A1, '
                     "12 November 2026).")

DROP_LEVEL = 0  # 1: RT2 without OCTID and OCTDL (its nine-way table is then not scored)
RULE_COMMIT = "0000000000000000000000000000000000000000"  # the commit that fixed the approved re-test rule (protocols.md section 13)


def job_id(name, arm, variant):
    return "-".join(filter(None, (name, arm, variant)))

In [ ]:
# Settings and helpers
import csv, datetime, hashlib, json, os, pathlib, shlex, shutil, subprocess, sys, time, zipfile

COMMIT = "0000000000000000000000000000000000000000"  # the locked code; change only on purpose
REPO = "https://github.com/Malnoad/Few-shot-rare-diseases"
ROOT, ENV = "/content/Few-shot", "/content/fm-env"
PY = f"{ENV}/bin/python"
PREPARED, ASSETS = "/content/prepared", "/content/assets"  # PREPARED becomes the input-lock folder at Step 12
WORK = "/content/fm_v1"  # specs, ledger, pilots, logs, reports, external query lock and features
LEDGER, REPORTS = f"{WORK}/download_ledger.csv", f"{WORK}/reports"
FEATURES, EXTERNAL_LOCK, META = f"{WORK}/features", f"{WORK}/external", f"{WORK}/external_meta"
DOWNLOADS, EXPORTS = "/content/external_downloads", "/content/fm_exports"  # big downloads; ZIPs made for Drive
LOCK_ZIP = f"{EXPORTS}/input-lock.zip"
RAW = "/content/drive/MyDrive/octfsl/raw"  # Lam's folder, through the runner's octfsl shortcut (guide, Part 1)
OUT = "/content/drive/MyDrive/octfsl/fm_v1"  # Lam's output folder, through the same shortcut
IMAGES = 84630
ARMS = ("RETFound", "DINOv2", "MIRAGE", "Inception")  # three FMs and the reference (A3 S-4: no fallback)

# Produced on the laptop from the same code and manifest (9 Nov).
EXPECTED_PREPARED = {
    "groups.csv": "6991d8229149aadf4af5ee53f1e2eb3991805d3bf56ebb5b367337a30224c5d0",
    "nine_way_roster.csv": "72f90c00b98810af46d257fe53d047d93f5312552ebaa478aef72d58da81b586",
    "episodes_development.csv": "cd060bddb434b5de476f4fd3550b8b21481ebaf3eb597298f727517968544695",
    "episodes_reporting.csv": "f9e1e3c6a4f2aacd405171837d472190271da2b03dbce79b619a03900b202b8f",
    "timing_inventory.csv": "a6be7a5ec3ba1bbe220674faaef8e51105bc7e4da59e9672d20e4e4562be97ed",
    "extraction_inventory.csv": "5ca665e4828b2d45c7d90cc6363708d9548793de7019d417e57222ee2b76b44d",
}


def sh(command):
    """Run a shell command, streaming its output; any failure stops the notebook."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"FAILED: {command}")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def report(name, lines):
    """Print a report and keep a copy in REPORTS, so the save step can put it on Drive."""
    text = "\n".join(lines)
    print(text)
    path = pathlib.Path(REPORTS, name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text + "\n", encoding="utf-8")


def save_to_drive(folder, paths, replace=False):
    """Copy files into octfsl/fm_v1/<folder>, verify every copy, then disconnect Drive.
    A file already there with the same bytes is kept; different bytes are never overwritten,
    except a receipt saved with replace=True."""
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    try:
        target = pathlib.Path(OUT, folder)
        target.mkdir(parents=True, exist_ok=True)
        for path in map(pathlib.Path, paths):
            digest, copy = sha256(path), target / path.name
            if copy.exists() and not replace:
                assert sha256(copy) == digest, f"{copy} already exists with different bytes; nothing was overwritten"
            else:  # copy under a temporary name first, so an interrupted copy never blocks a retry
                pending = target / (path.name + ".part")
                shutil.copyfile(path, pending)
                assert sha256(pending) == digest, (f"The Drive copy of {path.name} differs: run this step again; if it fails "
                                                   f"again, download {path} from the Files panel and send it to Lam.")
                os.replace(pending, copy)
            print(f"saved {path.name}  {digest}")
    finally:
        drive.flush_and_unmount()

# One session; the reused Phase-1 cells read these two names.
SESSION, SESSION_JOBS = 1, {1: [('RT2', 'Inception', 'I-pil', 0),
 ('RT2', 'Inception', 'I-ref', 0),
 ('RT1', 'RETFound', 'R-ref', 0),
 ('RT1', 'RETFound', 'upstream-check', 0),
 ('RT1', 'RETFound', 'R-meanraw', 0),
 ('RT1', 'RETFound', 'R-clsnorm', 0),
 ('RT1', 'RETFound', 'R-meannorm', 0)]}

In [ ]:
MODEL_HEADS = {'RETFound': 'https://huggingface.co/YukunZhou/RETFound_mae_natureOCT/resolve/8e551e2f547d3e27aedf43c9cf1c5bcea1c17171/RETFound_mae_natureOCT.pth'}
# Step 0. CPU is enough. Check access to the locked models and Drive; do no extraction.
from google.colab import drive, userdata
import requests


def colab_token():
    try:
        return userdata.get("HF_TOKEN")
    except Exception as exc:
        print(f"HF_TOKEN secret could not be read ({type(exc).__name__}). "
              "Check Colab Secrets access and retry if the request timed out.")
        return None


def access_check(url, token=None):
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    try:
        with requests.head(url, headers=headers, timeout=30, allow_redirects=False) as response:
            return ("ok" if response.status_code in (200, 206) or
                    (response.is_redirect and response.headers.get("Location")) else f"HTTP {response.status_code}")
    except requests.RequestException as exc:
        return type(exc).__name__


def preflight():
    assert RUN_BY.strip() and not any(mark in RUN_BY for mark in ("<", "[", "\n")), "Fill in RUN_BY."
    imports = subprocess.run([sys.executable, "-E", "-s", "-c", "import tensorflow, scipy, PIL"],
                             capture_output=True, text=True)
    print("Step 0 imports (tensorflow, scipy, PIL):",
          "ok" if imports.returncode == 0 else f"failed (exit {imports.returncode})\n{imports.stderr.strip()}")
    token = colab_token()
    models = {
        arm: access_check(url, token if arm == "RETFound" else None)
        if arm != "RETFound" or token else "HF_TOKEN missing or notebook access off"
        for arm, url in MODEL_HEADS.items()
    }
    gpu = shutil.which("nvidia-smi")
    result = subprocess.run([gpu, "--query-gpu=name,compute_cap,memory.total,driver_version",
                             "--format=csv,noheader"], capture_output=True, text=True) if gpu else None
    runtime = result.stdout.strip() if result and result.returncode == 0 else "CPU"
    drive.mount("/content/drive", force_remount=True)
    try:
        assert pathlib.Path(RAW).is_dir() and pathlib.Path(OUT).is_dir(), "Add the octfsl My Drive shortcut."
        free = shutil.disk_usage(OUT).free
        assert free >= 6_000_000_000, "Drive requires at least 6 GB free."
        probe = pathlib.Path(OUT, f".retest-write-test-{time.time_ns()}")
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
    finally:
        drive.flush_and_unmount()
    print(f"RETEST PREFLIGHT | by={RUN_BY} | commit={COMMIT} | runtime={runtime} | "
          f"Drive free={free / 1e9:.1f} GB | imports={'ok' if imports.returncode == 0 else 'failed'} | "
          + " | ".join(f"{arm}={state}" for arm, state in models.items()))
    return runtime


RUNTIME = preflight()

In [ ]:
# Step 1. Select the plan and verify the GPU.
assert DROP_LEVEL in (0, 1), "Use DROP_LEVEL 0 or 1."
assert "A100" in RUNTIME or "L4" in RUNTIME, "Use an A100 or L4 GPU."
PHASE = pathlib.Path("/content/retest")
LOCK, EXTERNAL_LOCK = PHASE / "lock", PHASE / "external"
EXPORT, RENDER, PNG = PHASE / "exports", PHASE / "render", PHASE / "png"
PNG_READY = PHASE / "png.ok"
DRIVE_FOLDER = f"final/{ATTEMPT}/retest"
SCRIPT = f"{ROOT}/fm_retest_extract.py"
RT_ARMS = ("Inception", "RETFound")

In [ ]:
# Step 2. Get the exact code and build the same Python environment as at the lock
# Colab's own Python can change between sessions (3.13.16 at the lock, 3.13.18 on 14 Nov). The lock fixed
# Python 3.13.16 (input_lock.json, evaluation_environment), so this step installs that exact build,
# checked by SHA256 (python-build-standalone release 20261020), and builds the environment with it.
PYTHON_VERSION = "3.13.16"
PYTHON_URL = ("https://github.com/astral-sh/python-build-standalone/releases/download/20261020/"
              "cpython-3.13.16%2B20261020-x86_64-unknown-linux-gnu-install_only.tar.gz")
PYTHON_SHA256 = "0ad293ecda8ea93a28e4a6b5e9cc20ae74be15aafed75be04418b8f82bd0d0da"
BASE = f"/content/python-{PYTHON_VERSION}"
if not os.path.isdir(ROOT):
    sh(f"git clone -q {REPO} {ROOT}")
sh(f"git -C {ROOT} checkout -q {COMMIT}")
head = subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, head
print("code:", head)
assert pathlib.Path(SCRIPT).is_file(), "This code pin predates the re-test runner. Open the rebuilt notebook."


def python_version(python):
    return subprocess.check_output([python, "-c", "import platform; print(platform.python_version())"], text=True).strip()


if not os.path.exists(f"{BASE}/bin/python3"):
    tarball = f"{BASE}.tar.gz"
    sh(f"curl -sSfL -o {tarball} {PYTHON_URL}")
    assert sha256(tarball) == PYTHON_SHA256, "The Python download differs from the pinned build: run this step again; if it fails again, tell Lam."
    shutil.rmtree(f"{BASE}.part", ignore_errors=True)
    os.makedirs(f"{BASE}.part")
    sh(f"tar -xzf {tarball} -C {BASE}.part")
    os.replace(f"{BASE}.part/python", BASE)  # complete or absent, so a rerun never sees half a Python
    shutil.rmtree(f"{BASE}.part", ignore_errors=True)
assert python_version(f"{BASE}/bin/python3") == PYTHON_VERSION, "The installed Python is not the locked version: tell Lam."
if os.path.exists(PY) and python_version(PY) != PYTHON_VERSION:
    shutil.rmtree(ENV)  # an environment built earlier with Colab's own Python
if not os.path.exists(PY):
    sh(f"{BASE}/bin/python3 -m venv --without-pip {ENV}")
sh(f"python3 -m pip --python {PY} install -q -r {ROOT}/requirements-fm.txt")
sh(f"python3 -m pip --python {PY} check")
sh(f"""{PY} -c 'import sys, numpy, scipy; print("python", sys.version.split()[0], "numpy", numpy.__version__, "scipy", scipy.__version__)'""")

In [ ]:
# Step 2a. Rules before numbers: the pinned commit is on origin and holds the approved, pushed re-test rule.
contains = subprocess.check_output(["git", "-C", ROOT, "branch", "-r", "--contains", COMMIT], text=True)
branches = [line.strip() for line in contains.splitlines() if line.strip() and "->" not in line]
assert branches, "The pinned commit is on no origin branch. Stop and tell Lam."
headings = [line for line in pathlib.Path(ROOT, "protocols.md").read_text(encoding="utf-8").splitlines()
            if line.startswith("## 13. Re-tests RT1")]
assert len(headings) == 1 and "DRAFT" not in headings[0].upper(), (
    "The re-test rule (protocols.md section 13) is missing or still a DRAFT at this commit. Stop: no worker may run.")
added = subprocess.check_output(["git", "-C", ROOT, "log", "--reverse", "--format=%H", "-S", headings[0], "--",
                                 "protocols.md"], text=True).split()
assert added and added[0] == RULE_COMMIT, "RULE_COMMIT is not the commit that fixed the approved rule. Stop and tell Lam."
assert subprocess.run(["git", "-C", ROOT, "merge-base", "--is-ancestor", RULE_COMMIT, COMMIT]).returncode == 0, (
    "The rule commit is not part of the pinned code. Stop and tell Lam.")
RULE_LINE = f"commit {COMMIT[:7]} on {branches[0]} | rule {RULE_COMMIT[:7]}: {headings[0]}"
print("rule ok |", RULE_LINE)

In [ ]:
# Step 2b. PyTorch in the study environment sees the GPU (MIRAGE); Inception uses Colab's own TensorFlow, as at the lock
sh(f"""{PY} -c 'import torch; assert torch.cuda.is_available(), "PyTorch cannot see the GPU"; print("torch", torch.__version__, "cuda", torch.version.cuda)'""")
sh("""python3 -c 'import sys, tensorflow as tf; print("Colab python", sys.version.split()[0], "tensorflow", tf.__version__, "(the lock had 3.13.16 and 2.21.0; differences are recorded and measured by the control images)")'""")

In [ ]:
# Step 3. Check Drive, unpack the images, then disconnect Drive
from google.colab import drive


def drive_ready():
    """Check both folders now, so a Drive problem cannot surface only at Step 11."""
    for path in (RAW, OUT):
        assert os.path.isdir(path), f"{path} not found: add the octfsl shortcut to your My Drive (guide, Part 1)"
    probe = pathlib.Path(OUT, f".write-test-{time.time_ns()}")
    probe.write_text("ok", encoding="utf-8")
    assert probe.read_text(encoding="utf-8") == "ok", "octfsl/fm_v1 is not writable: ask Lam to check Editor access"
    probe.unlink()
    print("Drive ok: octfsl/raw is readable and octfsl/fm_v1 is writable")


def extract(zip_path, anchor):
    """Copy every file below the folder named `anchor` into data/raw, skipping macOS junk."""
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            parts = [p for p in info.filename.replace("\\", "/").split("/") if p]
            if not parts or info.filename.endswith(("/", "\\")) or "__MACOSX" in parts or parts[-1].startswith("._"):
                continue
            if anchor not in parts[:-1]:
                continue
            target = pathlib.Path(ROOT, "data/raw", *parts[parts.index(anchor):])
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
            count += 1
    print(f"{pathlib.Path(zip_path).name}: {count} files")


with open(f"{ROOT}/data/manifest.csv", newline="", encoding="utf-8") as stream:
    MANIFEST = [row["path"].replace("\\", "/") for row in csv.DictReader(stream)]


def absent():
    return [p for p in MANIFEST if not os.path.isfile(f"{ROOT}/{p}")]


assert len(MANIFEST) == IMAGES, len(MANIFEST)
if absent() or not globals().get("DRIVE_OK"):
    drive.mount("/content/drive", force_remount=True)
    drive_ready()
    if absent():
        extract(f"{RAW}/oct2017.zip", "OCT2017")
        extract(f"{RAW}/rare.zip", "rare")
    drive.flush_and_unmount()
    DRIVE_OK = True
missing_images = absent()
assert not missing_images, f"{len(missing_images)} manifest images missing; first: {missing_images[0]}"
print(len(MANIFEST), "manifest images present")

In [ ]:
# Step 4. Restore the locked input lock and external query lock from Drive (read-only copies)
from google.colab import drive
os.makedirs(f"{PHASE}/archives", exist_ok=True)
drive.mount("/content/drive", force_remount=True)
try:
    for name in ("input-lock.zip", "external-lock.zip"):
        shutil.copyfile(f"{OUT}/final/{ATTEMPT}/{name}", f"{PHASE}/archives/{name}")
finally:
    drive.flush_and_unmount()
assert sha256(f"{PHASE}/archives/input-lock.zip") == LOCK_SHA256, "input-lock.zip differs from the locked SHA256: stop and tell Lam."
assert sha256(f"{PHASE}/archives/external-lock.zip") == EXTERNAL_SHA256, "external-lock.zip differs from the locked SHA256: stop and tell Lam."
sh(f"{PY} -E -s {ROOT}/fm_prepare.py restore --root {ROOT} --out {PHASE}/lock --archive {PHASE}/archives/input-lock.zip --sha256 {LOCK_SHA256}")
with zipfile.ZipFile(f"{PHASE}/archives/external-lock.zip") as archive:
    archive.extractall(f"{PHASE}/external")
print("Locks restored:", LOCK_SHA256[:12], EXTERNAL_SHA256[:12])

In [ ]:
# Step 5a. Define the checkpoint downloads (revisions and digests come from fm_extract.py)
import requests

pins = json.loads(subprocess.check_output([PY, "-c", (
    "import json, fm_extract as x; print(json.dumps(dict(repositories=x.REPOSITORIES, revisions=x.REVISIONS, "
    "sources=x.SOURCE_REVISIONS, digests=x.EXPECTED_ASSET_SHA256)))")], cwd=ROOT, text=True))


def hf(arm, name):
    return f"https://huggingface.co/{pins['repositories'][arm]}/resolve/{pins['revisions'][arm]}/{name}"


def github(repo, arm, path):
    return f"https://raw.githubusercontent.com/{repo}/{pins['sources'][arm]}/{path}"


KERAS = "inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"
FILES = [  # arm, file name, source, needs the Hugging Face token
    ("RETFound", "RETFound_mae_natureOCT.pth", hf("RETFound", "RETFound_mae_natureOCT.pth"), True),
    ("RETFound", "models_vit.py", github("rmaphoh/RETFound_MAE", "RETFound", "models_vit.py"), False),
    ("DINOv2", "model.safetensors", hf("DINOv2", "model.safetensors"), False),
    ("DINOv2", "config.json", hf("DINOv2", "config.json"), False),
    ("DINOv2", "preprocessor_config.json", hf("DINOv2", "preprocessor_config.json"), False),
    ("MIRAGE", "model.safetensors", hf("MIRAGE", "model.safetensors"), False),
    ("MIRAGE", "config.json", hf("MIRAGE", "config.json"), False),
    ("MIRAGE", "mirage_hf.py", github("j-morano/MIRAGE", "MIRAGE", "hf/mirage_hf.py"), False),
    ("Inception", KERAS, f"https://storage.googleapis.com/tensorflow/keras-applications/inception_v3/{KERAS}", False),
]
assert {(arm, name) for arm, name, _, _ in FILES} == {(arm, name) for arm in ARMS for name in pins["digests"][arm]}


def record(row):
    """Append to the download-cost ledger the protocol asks for (runbook, R4)."""
    path = pathlib.Path(LEDGER)
    path.parent.mkdir(parents=True, exist_ok=True)
    new = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        if new:
            writer.writerow(["arm", "file", "url", "bytes", "seconds", "utc"])
        writer.writerow(row)


def fetch(arm, name, url, token=None):
    """Download one pinned file. Bytes that fail the digest are deleted, never kept."""
    target = pathlib.Path(ASSETS, arm, name)
    expected = pins["digests"][arm][name]
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and sha256(target) == expected:
        print(f"already verified  {arm}/{name}")
        return
    partial = target.with_name(name + ".part")
    digest, size, start = hashlib.sha256(), 0, time.time()
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    with requests.get(url, headers=headers, stream=True, timeout=60) as response:
        response.raise_for_status()
        with partial.open("wb") as stream:
            for chunk in response.iter_content(1 << 20):
                stream.write(chunk)
                digest.update(chunk)
                size += len(chunk)
    seconds = round(time.time() - start, 1)
    if digest.hexdigest() != expected:
        partial.unlink()
        raise ValueError(f"{arm}/{name} does not match its pinned SHA256; nothing was kept")
    partial.replace(target)
    record([arm, name, url, size, seconds, datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")])
    print(f"downloaded {size / 1e6:9.1f} MB in {seconds:7.1f} s  {arm}/{name}")

In [ ]:
# Step 5b. Stage pinned weights. A download failure in one arm leaves the others runnable.
from collections import defaultdict

asset_issues = {}
FILES = [row for row in FILES if row[0] in RT_ARMS]  # RETFound and Inception-v3 only
assets_by_arm = defaultdict(list)
for arm, name, url, gated in FILES:
    assets_by_arm[arm].append((name, url, gated))
for arm, assets in assets_by_arm.items():
    try:
        for name, url, gated in assets:
            token = colab_token() if gated else None
            if gated and not token:
                raise ValueError("HF_TOKEN missing or notebook access off")
            fetch(arm, name, url, token)
    except (OSError, ValueError, requests.RequestException) as exc:
        asset_issues[arm] = f"weights could not be staged ({type(exc).__name__})"
        print(f"{arm}: {asset_issues[arm]}. Rerun this cell once unchanged if needed.")
    finally:
        token = None

In [ ]:
# Step 6a. Download the external datasets (images only: no encoder, no label used, no score)
BOREALIS = "https://borealisdata.ca/api"
OCTID = {"doi:10.5683/SP/WLW4ZT": "NORMAL", "doi:10.5683/SP/MBMQGD": "MH", "doi:10.5683/SP/FVLZIL": "CSR"}
OCTID_LISTING_SHA256 = "984fde2b5222c44a3c2ed7bf34b5d513a921b7d88d3a12e4518cbb28f52b6f21"  # checked 10 Nov
OCTID_IMAGES = {"NORMAL": 206, "MH": 102, "CSR": 102}
ZENODO = "https://zenodo.org/records/10839556/files"
OCTDL_MD5 = {"OCTDL.zip": "dec9766fc916202c45a333c4f5a09576", "OCTDL_labels.csv": "2fced789038c6ca83283066c2f8ad289"}
OCTDL_IMAGES = {"NO": 332, "VID": 76}
A1_COUNTS = {"OCTDL/MH": 39, "OCTDL/NORMAL": 332, "OCTID/CSC": 102, "OCTID/MH": 102, "OCTID/NORMAL": 206}  # amendment A1
EXTERNAL = pathlib.Path(ROOT, "data/external")
for folder in (META, DOWNLOADS):
    os.makedirs(folder, exist_ok=True)


def md5(path):
    digest = hashlib.md5()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_md5(source, url, target, expected):
    """Download one public file. Bytes that fail the published MD5 are deleted, never kept."""
    target = pathlib.Path(target)
    if target.exists() and md5(target) == expected:
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    partial, size, start = target.with_name(target.name + ".part"), 0, time.time()
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with partial.open("wb") as stream:
            for chunk in response.iter_content(1 << 20):
                stream.write(chunk)
                size += len(chunk)
    if md5(partial) != expected:
        partial.unlink()
        raise ValueError(f"{target.name} does not match its published MD5; nothing was kept")
    partial.replace(target)
    record([source, target.name, url, size, round(time.time() - start, 1), datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")])


listing = []
for doi in OCTID:
    response = requests.get(f"{BOREALIS}/datasets/:persistentId/", params={"persistentId": doi}, timeout=120)
    response.raise_for_status()
    version = response.json()["data"]["latestVersion"]
    title = next(field["value"] for field in version["metadataBlocks"]["citation"]["fields"] if field["typeName"] == "title")
    listing += [[doi, title, f["dataFile"]["filename"], f["dataFile"]["filesize"], f["dataFile"]["md5"], f["dataFile"]["id"]] for f in version["files"]]
listing.sort(key=lambda row: (row[0], row[2]))
with open(f"{META}/octid_files.csv", "w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream, lineterminator="\n")
    writer.writerow(["doi", "title", "filename", "bytes", "md5", "file_id"])
    writer.writerows(listing)
assert sha256(f"{META}/octid_files.csv") == OCTID_LISTING_SHA256, "The OCTID release changed since 10 Nov: stop and report."
for doi, _, name, _, digest, file_id in listing:
    if name.lower().endswith(".jpeg"):
        fetch_md5("OCTID", f"{BOREALIS}/access/datafile/{file_id}", EXTERNAL / "OCTID" / OCTID[doi] / name, digest)
found = {folder: len(list((EXTERNAL / "OCTID" / folder).glob("*.jpeg"))) for folder in OCTID_IMAGES}
assert found == OCTID_IMAGES, found

fetch_md5("OCTDL", f"{ZENODO}/OCTDL.zip?download=1", f"{DOWNLOADS}/OCTDL.zip", OCTDL_MD5["OCTDL.zip"])
fetch_md5("OCTDL", f"{ZENODO}/OCTDL_labels.csv?download=1", f"{META}/OCTDL_labels.csv", OCTDL_MD5["OCTDL_labels.csv"])
with zipfile.ZipFile(f"{DOWNLOADS}/OCTDL.zip") as archive:
    for info in archive.infolist():
        parts = info.filename.split("/")
        if not info.is_dir() and len(parts) == 3 and parts[0] == "OCTDL" and parts[1] in OCTDL_IMAGES:
            target = EXTERNAL.joinpath(*parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
found = {folder: len(list((EXTERNAL / "OCTDL" / folder).glob("*.jpg"))) for folder in OCTDL_IMAGES}
assert found == OCTDL_IMAGES, found

In [ ]:
# Step 6b. Keep Drive and credentials away from the model code (risk record R1)
TOKEN_CACHES = ("/root/.cache/huggingface/token", "/root/.cache/huggingface/stored_tokens", "/root/.huggingface/token")
assert not os.path.ismount("/content/drive"), "Drive is mounted: run drive.flush_and_unmount() first."
cached = [path for path in TOKEN_CACHES if os.path.exists(path)]
assert not cached, f"A Hugging Face login is cached at {cached}. Use a fresh runtime, so the model code cannot read it."
SAFE_ENV = {name: value for name, value in os.environ.items()
            if not any(word in name.upper() for word in ("TOKEN", "SECRET", "KEY", "PASSW", "CREDENTIAL", "HF_", "HUGGING"))}
print("Removed from the worker environment (names only):", sorted(set(os.environ) - set(SAFE_ENV)) or "nothing")
INTERPRETER = {**{arm: PY for arm in ARMS}, "Inception": sys.executable}  # the reference uses Colab's own TensorFlow


def worker(command, log):
    """Run one process with no credentials in its environment, keep its output in `log`, wait until it exits."""
    process = subprocess.Popen(command, env=SAFE_ENV, cwd="/content", stdin=subprocess.DEVNULL,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    os.makedirs(os.path.dirname(log), exist_ok=True)
    with open(log, "w", encoding="utf-8") as stream:
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
    return process.pid, process.wait()


def gpu_users():
    return subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True).strip()

In [ ]:
# Step 6c. The checks in one line, before any worker: code and rule, both locks, the weights.
for name, expected in (("input-lock.zip", LOCK_SHA256), ("external-lock.zip", EXTERNAL_SHA256)):
    assert sha256(f"{PHASE}/archives/{name}") == expected, f"{name} differs from the locked SHA256: stop and tell Lam."
for arm, name, _, _ in FILES:
    if arm not in asset_issues:
        assert sha256(f"{ASSETS}/{arm}/{name}") == pins["digests"][arm][name], f"{arm}/{name} differs from its pinned SHA256."
print(f"RETEST CHECKS | {RULE_LINE} | locks {LOCK_SHA256[:12]} {EXTERNAL_SHA256[:12]} | weights "
      + ", ".join(f"{arm}={asset_issues.get(arm, 'verified')}" for arm in RT_ARMS))

In [ ]:
# Step 7. Verify previous exports and prepare immediate saves.
selected = [row for row in SESSION_JOBS[SESSION] if not row[3] or DROP_LEVEL < row[3]]


def archive_name(key):
    return f"retest-{key}.zip"


def receipt_path(archive):
    return archive.with_suffix(".zip.json")


saved = {}
pending_saves = globals().get("pending_saves", {})
EXPORT.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive", force_remount=True)
try:
    folder = pathlib.Path(OUT, DRIVE_FOLDER)
    keys = ["render", *(job_id(name, arm, variant) for name, arm, variant, _ in selected)]
    for key in keys:
        remote = folder / archive_name(key)
        receipt = receipt_path(remote)
        if remote.is_file() != receipt.is_file():
            orphan = remote if remote.is_file() else receipt
            orphan.unlink()
            print(f"{key}: removed unpaired Drive file {orphan.name}")
        elif remote.is_file():
            expected = json.loads(receipt.read_text(encoding="utf-8"))["sha256"]
            assert sha256(remote) == expected, f"{remote.name} differs from its Drive receipt."
            saved[key] = expected
            if key == "render" and not (RENDER / "complete.json").is_file():
                local = EXPORT / remote.name
                shutil.copyfile(remote, local)
                assert sha256(local) == expected, "Local render archive differs from its Drive receipt."
                pending = PHASE / "render-restore"
                pending.mkdir(exist_ok=True)
                with zipfile.ZipFile(local) as archive:
                    members = archive.namelist()
                    assert len(members) == len(set(members)) and "complete.json" in members
                    assert all(not pathlib.PurePosixPath(name).is_absolute() and
                               ".." not in pathlib.PurePosixPath(name).parts for name in members)
                    archive.extractall(pending)
                assert not RENDER.exists(), "The local render folder is incomplete. Use a fresh runtime."
                pending.rename(RENDER)
finally:
    drive.flush_and_unmount()


def save_output(key, output):
    archive = EXPORT / archive_name(key)
    receipt = receipt_path(archive)
    if not archive.is_file() or not receipt.is_file():
        sh(shlex.join([PY, "-E", "-s", f"{ROOT}/fm_prepare.py", "export", "--root", ROOT,
                       "--out", str(output), "--archive", str(archive)]))
    expected = json.loads(receipt.read_text(encoding="utf-8"))["sha256"]
    assert sha256(archive) == expected, f"{archive.name} differs from its local receipt."
    save_to_drive(DRIVE_FOLDER, [receipt, archive])
    saved[key] = expected


def try_save(key, output):
    try:
        save_output(key, output)
    except Exception as exc:
        pending_saves[key] = output
        message = f"{key}: Drive save failed ({type(exc).__name__}: {exc})"
        print(message, flush=True)
        logs = PHASE / "logs"
        logs.mkdir(parents=True, exist_ok=True)
        with (logs / f"{key}-save.log").open("a", encoding="utf-8") as log:
            log.write(message + "\n")
    else:
        pending_saves.pop(key, None)

In [ ]:
# Step 8. RT2 inputs (CPU): every image decoded by PIL and saved as PNG; only the receipts are archived.
def run_step(key, command):
    """Run at most twice unchanged in this runtime and keep both logs."""
    assert not os.path.ismount("/content/drive"), "Unmount Drive before running workers."
    logs = PHASE / "logs"
    for attempt in (1, 2):
        log = logs / f"{key}-try{attempt}.log"
        if not log.exists():
            print(f"{key}: attempt {attempt}", flush=True)
            _, code = worker(command, str(log))
            return code == 0
    print(f"{key}: two attempts recorded; see the logs.")
    return False



if not PNG_READY.is_file():  # also rebuilds the PNGs when the receipts were restored from Drive, and compares them
    assert run_step("render", [PY, "-E", "-s", SCRIPT, "render", "--root", ROOT, "--lock", str(LOCK),
                               "--external", str(EXTERNAL_LOCK), "--out", str(RENDER), "--png", str(PNG)]), "See render logs."
    PNG_READY.write_text("ok\n", encoding="utf-8")
if "render" not in saved:
    try_save("render", RENDER)

In [ ]:
# Step 9. One set or read-out per process: RT2 first, then RT1. Save each result before continuing.
assert PNG_READY.is_file(), "Finish the render step before extraction."
for name, arm, variant, _ in selected:
    key = job_id(name, arm, variant)
    if key in saved:
        print(f"{key}: verified on Drive")
        continue
    if arm in asset_issues:
        print(f"{key}: {asset_issues[arm]}")
        continue
    output = PHASE / "out" / name / variant
    if not (output / "complete.json").is_file():
        command = [INTERPRETER[arm], "-E", "-s", SCRIPT, "extract", "--root", ROOT, "--lock", str(LOCK),
                   "--external", str(EXTERNAL_LOCK), "--variant", variant, "--assets", f"{ASSETS}/{arm}",
                   "--out", str(output)]
        if variant == "I-pil":
            command += ["--render", str(RENDER), "--png", str(PNG)]
        if name == "RT2" and DROP_LEVEL:
            command.append("--main-only")
        if not run_step(key, command):
            tried_twice = (PHASE / "logs" / f"{key}-try2.log").is_file()
            print(f"{key}: worker failed twice; report the logs." if tried_twice else
                  f"{key}: worker failed. Rerun this cell unchanged to use the single retry.")
            continue
    try_save(key, output)

In [ ]:
# Step 10. Save logs and stamp, including on a partial run.
for key, output in list(globals().get("pending_saves", {}).items()):
    try_save(key, output)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
jobs = SESSION_JOBS[SESSION]
keys = [job_id(name, arm, variant) for name, arm, variant, tier in jobs if not tier or DROP_LEVEL < tier]
saved = globals().get("saved", {})
missing = (["render"] if "render" not in saved else []) + [key for key in keys if key not in saved]
dropped = [job_id(name, arm, variant) for name, arm, variant, tier in jobs if tier and tier <= DROP_LEVEL]
completed = ", ".join(saved) or "none"
line = (f"RETEST STAMP | {stamp} | drop_level={DROP_LEVEL} | by={RUN_BY} | GPU={RUNTIME} | commit={COMMIT} | rule={RULE_COMMIT} | "
        f"completed={completed} | missing={', '.join(missing) or 'none'} | dropped={', '.join(dropped) or 'none'}")
logs = EXPORT / f"retest-logs-{stamp}.zip"
EXPORT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(logs, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((PHASE / "logs").glob("*.log")):
        archive.write(path, path.name)
    if pathlib.Path(LEDGER).is_file():
        archive.write(LEDGER, "download_ledger.csv")
    archive.writestr("stamp.txt", line + "\n")
    archive.writestr("asset_issues.json", json.dumps(globals().get("asset_issues", {}), indent=2) + "\n")
receipt = logs.with_suffix(".zip.json")
receipt.write_text(json.dumps({"sha256": sha256(logs)}, indent=2) + "\n", encoding="utf-8")
print(line)
print("Send this line to the agents. Scoring runs once, on the laptop (fm_posthoc.py rt).")
save_to_drive(DRIVE_FOLDER, [receipt, logs])